# Part 1 · Project structure, settings and schemas

The product of this session is a **FastAPI service** in [`app/`](../app/): the Research Brief Agent behind an HTTP API,
plus a demo chat UI. The four notebooks walk through it part by part; each part adds one layer of the final service:

| Part | Layer | Files |
| --- | --- | --- |
| **1** (this notebook) | project layout, settings, request/response schemas, the app object | `core/config.py`, `schemas.py`, `main.py` |
| 2 | the agent and its run streamed as Server-Sent Events | `agent/`, `services/streaming.py`, `routers/runs.py` |
| 3 | who is calling, whose thread it is, human approval over HTTP | `core/security.py`, `dependencies.py`, `services/threads.py` |
| 4 | errors, health checks, the demo UI | `core/errors.py`, `routers/health.py`, `static/` |

Each section shows a function's source, calls it, and prints what it returns. The full flow runs against the
**running service**, so start it first, in a terminal in the session folder (keep it open: its log shows each request,
run and tool call):
```bash
export UV_PROJECT_ENVIRONMENT=venvAE09        # Windows PowerShell: $env:UV_PROJECT_ENVIRONMENT="venvAE09"
uv run fastapi run --host 127.0.0.1
```

Docs: [Bigger Applications](https://fastapi.tiangolo.com/tutorial/bigger-applications/) ·
[Settings](https://fastapi.tiangolo.com/advanced/settings/) · [Request body](https://fastapi.tiangolo.com/tutorial/body/) ·
[Lifespan](https://fastapi.tiangolo.com/advanced/events/) · [FastAPI CLI](https://fastapi.tiangolo.com/fastapi-cli/)

In [1]:
import logging

from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
load_dotenv()

from utils import PROJECT_DIR, check_server, client, source, tree  # notebooks/utils.py

check_server()  # GET /health; fails fast with the start command

server: ok


## 1. The layout: one file per responsibility

A single `server.py` mixes configuration, schemas, auth, the agent and the routes; every change touches the same file
and nothing can be reused or tested alone. FastAPI's *Bigger Applications* guide splits an app into a package: the
**routers** hold the path operations (`APIRouter`), shared checks are **dependencies**, and `main.py` only assembles
them. This service adds `core/` (settings, security, errors), `services/` (logic that is not HTTP) and `agent/`.

Requests flow one way: `routers` → `dependencies` → `services` → `agent`. Nothing in `services/` or `agent/` imports
FastAPI request objects, so the same functions can be called from a notebook, a test or a worker.

In [2]:
tree()

app/agent/graph.py               The Research Brief Agent: one Deep Agent with web search and an approval step be
app/agent/prompts/research_brief.md 
app/agent/tools.py               Tools of the Research Brief Agent (copied from Session 06 and trimmed). Tools re
app/core/config.py               Settings read once from environment variables and the session `.env` (pydantic-s
app/core/errors.py               One error body for every failure: RFC 9457 Problem Details (`application/problem
app/core/security.py             Authentication: who is calling. The demo tokens stand in for an identity provide
app/dependencies.py              Dependencies shared by the routers: the objects created at startup, and the thre
app/main.py                      The FastAPI application: startup (lifespan), routers, error handlers and the dem
app/routers/health.py            Liveness (/health: the process answers) and readiness (/ready: it can serve a ru
app/routers/runs.py              Runs: send a messa

## 2. Settings: configuration from the environment

`Settings` (pydantic-settings) reads environment variables and the session `.env` once, with types and defaults:
`openai_api_key` is required, so the app refuses to start without it, and keys are `SecretStr`, so they never
appear in logs or reprs. `get_settings()` is cached: every module gets the same object.

In [3]:
from app.core.config import Settings, get_settings

source(Settings)

class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=PROJECT_DIR / ".env", extra="ignore")

    openai_api_key: SecretStr = Field(min_length=1)  # required and not empty: the app does not start without it
    tavily_api_key: SecretStr = Field(min_length=1)
    model: str = "openai:gpt-6-luna"
    run_timeout_s: int = 180  # a brief run normally takes 20-60 s
    users_file: Path = PROJECT_DIR / "data" / "users.json"
    data_dir: Path = PROJECT_DIR / "output"  # SQLite databases and saved briefs

In [4]:
settings = get_settings()
print("model:          ", settings.model)
print("run_timeout_s:  ", settings.run_timeout_s)
print("openai_api_key: ", repr(settings.openai_api_key))  # SecretStr hides the value
print("users_file:     ", settings.users_file.relative_to(PROJECT_DIR))
print("data_dir:       ", settings.data_dir.relative_to(PROJECT_DIR))

model:           openai:gpt-6-luna
run_timeout_s:   180
openai_api_key:  SecretStr('**********')
users_file:      data/users.json
data_dir:        output


Change a value without touching code: an environment variable wins over `.env`.

In [5]:
import os

os.environ["RUN_TIMEOUT_S"] = "60"
print("run_timeout_s with RUN_TIMEOUT_S=60:", Settings().run_timeout_s)
del os.environ["RUN_TIMEOUT_S"]

run_timeout_s with RUN_TIMEOUT_S=60: 60


## 3. Schemas: what a client may send and receives

The bodies are Pydantic models in [`app/schemas.py`](../app/schemas.py). FastAPI validates each request against them
before the route runs, and turns them into the OpenAPI schema. `RunRequest` forbids extra fields: the user is never
read from the body (Part 3).

In [6]:
from pydantic import ValidationError

from app.schemas import RunRequest, Thread

source(RunRequest)

class RunRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")  # no user_id here: the user comes from the token
    message: str = Field(min_length=1, max_length=4000, examples=["Brief on how well heat pumps work below 0°C."])

In [7]:
print("valid:", RunRequest(message="Brief on heat pumps below 0°C."))
for bad in ({"message": ""}, {"message": "hi", "user_id": "alice"}, {"text": "hi"}):
    try:
        RunRequest(**bad)
    except ValidationError as exc:
        print(f"\n{bad} ->", [(e["loc"], e["msg"]) for e in exc.errors()])

valid: message='Brief on heat pumps below 0°C.'

{'message': ''} -> [(('message',), 'String should have at least 1 character')]

{'message': 'hi', 'user_id': 'alice'} -> [(('user_id',), 'Extra inputs are not permitted')]

{'text': 'hi'} -> [(('message',), 'Field required'), (('text',), 'Extra inputs are not permitted')]


In [8]:
print(Thread(thread_id="9fa034ac2308", title="EV batteries", created_at="2026-11-01T13:30:29+00:00").model_dump_json(indent=1))

{
 "thread_id": "9fa034ac2308",
 "title": "EV batteries",
 "created_at": "2026-11-01T13:30:29+00:00",
 "status": "idle"
}


## 4. The app object: lifespan, routers, handlers

`main.py` builds the `FastAPI` object. `lifespan` runs once at startup (settings, model, checkpointer, store, agent)
and closes the databases at shutdown; `include_router` adds each router; `add_error_handlers` makes every error a
Problem Details body (Part 4). The FastAPI CLI finds the app through `[tool.fastapi] entrypoint = "app.main:app"`
in `pyproject.toml`:

| Command | Use |
| --- | --- |
| `uv run fastapi dev` | development: auto-reload on every file change (a reload stops runs that are streaming) |
| `uv run fastapi run --host 127.0.0.1` | serving: no reload; Uvicorn underneath. One worker process, because the registry of running threads is in memory |

In [9]:
from app import main

source(main.lifespan)

@asynccontextmanager
async def lifespan(app: FastAPI):
    """Created once before the first request, closed after the last one."""
    settings = get_settings()  # fails fast if a required key is missing
    os.environ.setdefault("OPENAI_API_KEY", settings.openai_api_key.get_secret_value())  # read by init_chat_model
    settings.data_dir.mkdir(exist_ok=True)
    async with (AsyncSqliteSaver.from_conn_string(str(settings.data_dir / "checkpoints.db")) as checkpointer,
                AsyncSqliteStore.from_conn_string(str(settings.data_dir / "threads.db")) as store):
        await store.setup()
        app.state.agent = build_agent(init_chat_model(settings.model), checkpointer)
        app.state.store = store
        app.state.running = {}  # thread_id -> run_id; in memory, so serve with ONE worker process
        logger.info("model %s · agent ready", settings.model)
        yield
    logger.info("shutdown: checkpointer and store closed")

In [10]:
spec = main.app.openapi()  # the OpenAPI document FastAPI builds from the routers (no server needed)
for path, operations in spec["paths"].items():
    for method, op in operations.items():
        print(f"{method.upper():<7}{path:<34}{op['tags'][0]:<9}{op['summary']}")

GET    /health                           health   Health
GET    /ready                            health   Ready
GET    /me                               threads  Me
GET    /threads                          threads  List Threads
POST   /threads                          threads  Create Thread
GET    /threads/{thread_id}              threads  Get Thread
POST   /threads/{thread_id}/runs         runs     Create Run
POST   /threads/{thread_id}/resume       runs     Resume Run


The running service publishes the same routes as an OpenAPI document (`/openapi.json`) and two UIs built from it:
**Swagger UI** at `/docs` (the *Authorize* button takes a token from `data/users.json`) and **ReDoc** at `/redoc`.
Open `/docs` in the browser now.

In [11]:
live = client(None).get("/openapi.json").json()
print(live["info"]["title"], live["info"]["version"], "·", len(live["paths"]), "paths · same as the app object:",
      live["paths"].keys() == spec["paths"].keys())
print("RunRequest in the published schema:", live["components"]["schemas"]["RunRequest"]["required"],
      "additionalProperties =", live["components"]["schemas"]["RunRequest"]["additionalProperties"])

Research Brief Agent API 1.0.0 · 7 paths · same as the app object: True
RunRequest in the published schema: ['message'] additionalProperties = False
